In [1]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model


In [ ]:
import cv2
import numpy as np
import os
from datetime import datetime

# Chemin vers les fichiers nécessaires
yolov3_weights_path = r'C:\Users\Salma\Desktop\yolo\yolov3\yolov3-wider_16000.weights'
yolov3_config_path = r'C:\Users\Salma\Desktop\yolo\yolov3\yolov3-face.cfg'
names_path = r'C:\Users\Salma\Desktop\yolo\yolov3\face.names'
output_folder = r'C:\Users\Salma\Desktop\yolo\yolov3\detected_faces'

# Créer le dossier de sortie s'il n'existe pas
os.makedirs(output_folder, exist_ok=True)

# Charger les noms des classes (généralement juste "face")
with open(names_path) as f:
    classes = [line.strip() for line in f.readlines()]

# Charger le réseau YOLO pour la détection de visage
net = cv2.dnn.readNet(yolov3_weights_path, yolov3_config_path)

# Fonction pour vérifier la proximité des visages déjà détectés
def is_nearby(new_face_center, detected_faces_centers, threshold=50):
    for center in detected_faces_centers:
        distance = np.linalg.norm(np.array(new_face_center) - np.array(center))
        if distance < threshold:
            return True
    return False

# Détection des visages dans une image
def detect_faces_in_image(image_path):
    # Liste pour stocker les centres des visages déjà détectés
    detected_faces_centers = []

    # Lire l'image
    frame = cv2.imread(image_path)
    height, width = frame.shape[:2]

    # Préparation de l'image pour YOLO
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)

    # Récupération des noms des couches de sortie
    layer_names = net.getLayerNames()
    output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

    # Détection des objets
    outputs = net.forward(output_layers)

    # Analyse des détections
    conf_threshold = 0.5
    nms_threshold = 0.4

    face_count = 0  # Compteur pour les visages détectés

    for output in outputs:
        for detection in output:
            scores = detection[5:]
            confidence = max(scores)
            if confidence > conf_threshold:
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)
                x = int(center_x - w / 2)
                y = int(center_y - h / 2)

                # Vérifier les dimensions valides du visage
                if x >= 0 and y >= 0 and w > 0 and h > 0:
                    face = frame[y:y+h, x:x+w]
                    face_center = (center_x, center_y)

                    # Vérifier que l'image de visage n'est pas vide et non déjà détectée
                    if face is not None and face.size != 0 and not is_nearby(face_center, detected_faces_centers):
                        # Nommer le fichier d'image avec le timestamp actuel et un compteur
                        face_filename = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{face_count}.jpg"
                        face_path = os.path.join(output_folder, face_filename)

                        # Enregistrer l'image du visage
                        cv2.imwrite(face_path, face)

                        # Ajouter le centre du visage à la liste des centres détectés
                        detected_faces_centers.append(face_center)

                        # Afficher et annoter l'image
                        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                        cv2.putText(frame, "", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                        print(f"Detected and saved: {face_filename}")

                        face_count += 1  # Incrémenter le compteur de visages

    # Afficher le résultat avec les visages détectés
    cv2.imshow('Face Detection', frame)
    cv2.waitKey(0)  # Attendre une touche pour fermer l'image affichée
    cv2.destroyAllWindows()

# Chemin vers l'image à analyser
image_path = r'C:\Users\Salma\Pictures\pfe\totale.jpg'

# Appeler la fonction pour détecter et enregistrer les visages dans l'image
detect_faces_in_image(image_path)


In [ ]:

train_loss, train_accuracy = model.evaluate(train_ds)
print("Train Loss:", train_loss)
print("Train Accuracy:", train_accuracy)

# Calculer l'accuracy et la loss sur les données de validation
val_loss, val_accuracy = model.evaluate(val_ds)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense

# Check the directory train
train_dir = r"C:\Users\Salma\Desktop\yolo\data\train"
print("Checking directory structure...")
for root, dirs, files in os.walk(train_dir):
    print(root, "contains", len(files), "files")

# Initialize ImageDataGenerator
generator = ImageDataGenerator(validation_split=0.2)  # Added validation split

# Flow from directory
train_ds = generator.flow_from_directory(train_dir, target_size=(224, 224), batch_size=32, subset='training')
val_ds = generator.flow_from_directory(train_dir, target_size=(224, 224), batch_size=32, subset='validation')
classes = list(train_ds.class_indices.keys())

# Build the model
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Conv2D(96, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(len(classes), activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# Training
history = model.fit(train_ds, epochs=30, validation_data=val_ds)

# Save the model
model.save('model_p.h5')

# Load the model
model = tf.keras.models.load_model('model_p.h5')
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.show()
